# 2024&ndash;2026 Structural Shifts in the European Grid

Historical reproduction (see the [bertellis-flow-diagnostics](../bertellis-flow-diagnostics/bertellis_flow_diagnostics.ipynb) notebook) only proves a platform can look backward at known moments. This notebook points the same reusable functions &ndash; `hourly_border_counts`, `flow_type_summary`, `convergence_summary` &ndash; at real, ongoing structural changes to how the European grid is coupled, to see whether they show up mechanically in the data.

Three verified events, each with a source, each queried the same way:

- **Baltic states synchronisation, 2025-02-07/09** &ndash; Estonia, Latvia, and Lithuania desynchronised from Russia's BRELL ring and synchronised with Continental Europe. The sharpest, most discrete break of the three. ([ENTSO-E announcement](https://www.entsoe.eu/news/2025/02/09/entso-e-confirms-successful-synchronization-of-the-continental-european-electricity-system-with-the-systems-of-the-baltic-countries/))
- **Flamanville 3 EPR grid connection, 2024-12-21** &ndash; France's first new nuclear unit in 25 years (1,630 MW) connected to the grid, ramping up through 2025. A gradual shift, not a step change. ([EDF press release](https://www.edf.fr/en/the-edf-group/dedicated-sections/journalists/all-press-releases/update-on-the-flamanville-epr-the-reactor-produces-its-first-electrons-on-the-national-electricity-grid))
- **Core "Advanced Hybrid Coupling" go-live, 2026-06-10/11** &ndash; a market-coupling methodology change on Core CCR's external borders. ([JAO confirmation](https://www.jao.eu/news/confirmation-core-advanced-hybrid-coupling-go-live))

Each window is 8 days (event date &plusmn; 4 days) at hourly resolution &ndash; wide enough to see a trend, narrow enough that the API doesn't auto-downgrade past hourly.

**Data source:** [ENTSO-E Transparency Platform](https://transparency.entsoe.eu), via the [Voltpath API](https://voltpath.ai) (docs at [docs.voltpath.ai](https://docs.voltpath.ai)). Physical flow data is CC-BY 4.0; day-ahead prices are reused under ENTSO-E's general Transparency Platform terms.

**Before you run this:** sign up at [voltpath.ai](https://voltpath.ai) for a free personal API key (copy it any time from [voltpath.ai/settings](https://voltpath.ai/settings)), then copy `.env.example` to `.env` and paste it in as `VOLTPATH_API_KEY` (or leave `.env` empty and you'll be prompted).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import plotly.express as px
from common import (
    FLOW_COLORS,
    convergence_summary,
    fetch_flows,
    hourly_border_counts,
)

In [ ]:
EVENTS = [
    {"name": "Baltic sync", "date": "2025-02-08", "date_from": "2025-02-04", "date_to": "2025-02-12"},
    {"name": "Flamanville 3 connection", "date": "2024-12-21", "date_from": "2024-12-17", "date_to": "2024-12-25"},
    {"name": "Core Advanced Hybrid Coupling", "date": "2026-06-10", "date_from": "2026-06-06", "date_to": "2026-06-14"},
]

## Congested / counter-intuitive borders around each event

In [ ]:
for event in EVENTS:
    df = fetch_flows(event["date_from"], event["date_to"], resolution="1H")
    frames = []
    for flow_type in ("Congestion", "Counter-intuitive"):
        counts = hourly_border_counts(df, flow_type)
        counts["flow_type"] = flow_type
        frames.append(counts)
    combined = pd.concat(frames)

    fig = px.line(
        combined,
        x="datetime",
        y="n_borders",
        color="flow_type",
        title=f"{event['name']}: congested / counter-intuitive borders, {event['date_from']} to {event['date_to']}",
        labels={"n_borders": "Borders", "datetime": "Time", "flow_type": "Flow type"},
        color_discrete_map=FLOW_COLORS,
    )
    fig.add_vline(x=event["date"], line_dash="dash", annotation_text=event["name"])
    fig.show()

## Did coupling reliability actually change?

`convergence_summary` gives a time-share per border. Splitting each event's window at the event date and comparing the average `share_converged` before vs. after is a direct, mechanical answer to "did this event measurably change how often prices converged" &ndash; the same question ENTSO-E's Bidding Zone Configuration Technical Report answers by hand for past market-coupling go-lives (its 2025 edition attributes Croatia's and Austria's 2022 congestion-income swings to that year's Core FBMC go-live, pp.60-63 covering 2021-2023 &ndash; before this API's window, but the same kind of before/after question).

In [ ]:
def before_after_convergence(df: pd.DataFrame, event_date: str) -> pd.Series:
    before = df[df["datetime"] < event_date]
    after = df[df["datetime"] >= event_date]
    return pd.Series(
        {
            "share_converged_before": convergence_summary(before)["share_converged"].mean(),
            "share_converged_after": convergence_summary(after)["share_converged"].mean(),
        }
    )


for event in EVENTS:
    df = fetch_flows(event["date_from"], event["date_to"], resolution="1H")
    comparison = before_after_convergence(df, event["date"])
    print(event["name"])
    print(comparison.to_string())
    print()

## Takeaways

- The same functions used to reproduce a year-old published analysis (see [bertellis-flow-diagnostics](../bertellis-flow-diagnostics/bertellis_flow_diagnostics.ipynb)) turn a historical curiosity into a live monitoring tool: point them at a grid synchronisation, a new nuclear unit, or a market-coupling methodology change, and the before/after shift shows up the same way.
- This is a monitoring pattern, not a one-off: the same three calls work for any future structural change, including next week's, without modification.

---

Data: [ENTSO-E Transparency Platform](https://transparency.entsoe.eu), via the [Voltpath API](https://voltpath.ai).